In [ ]:
import sys
import subprocess

def pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    print("Installing:", " ".join(packages))
    subprocess.check_call(cmd)

pip_install(
    "kaggle",
    "kagglehub",
    "emoji",
    "vaderSentiment",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm",
    "joblib",
    "xgboost>=2.1.0",
    "transformers",
    "datasets",
    "accelerate",
    "gradio",
)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os
import glob
import kagglehub

path = kagglehub.dataset_download("kirti1210/sarcoji-dataset")
print("Data located at:", path)

csv_candidates = sorted(
    glob.glob(f"{path}/**/*.csv", recursive=True),
    key=lambda p: os.path.getsize(p),
    reverse=True,
)
print("CSV files found:", csv_candidates)

In [ ]:
import numpy as np
import pandas as pd

raw_df = pd.read_csv(csv_candidates[0])
print(f"Loaded {len(raw_df)} rows from {os.path.basename(csv_candidates[0])}")
print(f"Columns: {list(raw_df.columns)}")
display(raw_df.head())

REQUIRED_COLS = ["Sarcastic", "Text"]
missing = [c for c in REQUIRED_COLS if c not in raw_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Available: {list(raw_df.columns)}")

def normalize_binary_label(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer, float, np.floating)):
        return int(x)
    s = str(x).strip().lower()
    mapping = {
        "1": 1, "true": 1, "yes": 1, "sarcastic": 1,
        "0": 0, "false": 0, "no": 0, "non-sarcastic": 0, "not sarcastic": 0,
    }
    return mapping.get(s, np.nan)

work_df = raw_df.copy()
work_df["label"] = work_df["Sarcastic"].apply(normalize_binary_label)
work_df = work_df.dropna(subset=["label"])
work_df["label"] = work_df["label"].astype(int)
work_df["Text"] = work_df["Text"].astype(str)

print(f"\nAfter cleaning: {len(work_df)} rows")
print(f"Label distribution:\n{work_df['label'].value_counts().sort_index()}")
print(f"Class balance: {work_df['label'].mean():.2%} sarcastic")

In [ ]:
ESR_URLS = [
    "https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1048/Emoji_Sentiment_Data_v1.0.csv?isAllowed=y&sequence=8",
    "https://raw.githubusercontent.com/omkar-foss/emosent-py/master/emosent/data/Emoji_Sentiment_Data_v1.0.csv",
]

EXPECTED_ESR_COLS = [
    "emoji", "codepoint", "occurrences", "position",
    "negative", "neutral", "positive", "unicode_name", "unicode_block",
]

def normalize_emoji_char(e: str) -> str:
    if pd.isna(e):
        return ""
    return str(e).replace("\ufe0f", "")

lex = None
last_error = None

for url in ESR_URLS:
    try:
        tmp = pd.read_csv(url, header=None)
        first_val = str(tmp.iloc[0, 0]).strip().lower()
        if first_val in {"emoji", "char"}:
            tmp = tmp.iloc[1:].reset_index(drop=True)
        if tmp.shape[1] < 9:
            raise ValueError(f"Unexpected ESR shape from {url}: {tmp.shape}")
        tmp = tmp.iloc[:, :9].copy()
        tmp.columns = EXPECTED_ESR_COLS
        for col in ["occurrences", "position", "negative", "neutral", "positive"]:
            tmp[col] = pd.to_numeric(tmp[col], errors="coerce")
        tmp = tmp.dropna(subset=["emoji", "occurrences", "negative", "neutral", "positive"]).copy()
        tmp["emoji_norm"] = tmp["emoji"].astype(str).map(normalize_emoji_char)
        denom = tmp["occurrences"].replace(0, np.nan)
        tmp["neg_p"] = tmp["negative"] / denom
        tmp["neu_p"] = tmp["neutral"] / denom
        tmp["pos_p"] = tmp["positive"] / denom
        tmp["sentiment_score"] = tmp["pos_p"] - tmp["neg_p"]
        lex = tmp.drop_duplicates(subset=["emoji_norm"]).reset_index(drop=True)
        print("Loaded ESR from:", url)
        break
    except Exception as e:
        last_error = e

if lex is None:
    raise RuntimeError(f"Failed to load ESR lexicon. Last error: {last_error}")

emoji2score = dict(zip(lex["emoji_norm"], lex["sentiment_score"]))
print("Lexicon rows:", len(lex))
display(lex.head())


1. `tokenize_social_text` — preserves emojis as raw UTF-8 tokens (Methods 1 & 2)
2. `tokenize_social_text_demojized` — converts emojis to English descriptions first (Method 2b)


In [ ]:
import re
import emoji
from tqdm.auto import tqdm
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

tqdm.pandas()
vader = SentimentIntensityAnalyzer()

URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HTML_RE = re.compile(r"&\w+;|<[^>]+>")
WS_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"[#]?\w+(?:'\w+)?|[!?]+|[.,;:]+", flags=re.UNICODE)

def clean_social_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text

def extract_emoji_matches(text: str):
    return emoji.emoji_list(text)

def strip_emojis(text: str) -> str:
    return emoji.replace_emoji(text, replace=" ")

def tokenize_segment(segment: str):
    return WORD_RE.findall(segment)

def tokenize_social_text(text: str):
    text = clean_social_text(text)
    matches = extract_emoji_matches(text)
    if not matches:
        return tokenize_segment(text)
    tokens = []
    last = 0
    for m in matches:
        start, end, emj = m["match_start"], m["match_end"], m["emoji"]
        if start > last:
            tokens.extend(tokenize_segment(text[last:start]))
        tokens.append(normalize_emoji_char(emj))
        last = end
    if last < len(text):
        tokens.extend(tokenize_segment(text[last:]))
    return [tok for tok in tokens if tok and not tok.isspace()]

EMOJI_NAME_RE = re.compile(r":([a-zA-Z0-9_\-]+):")

def demojize_text(text: str) -> str:
    """Replace emoji characters with their English name tokens.

    Examples:
        🙄  → face with rolling eyes
        🔥  → fire
        🐻  → bear face
        ❤️  → red heart
    """
    demojized = emoji.demojize(text)
    result = EMOJI_NAME_RE.sub(
        lambda m: " " + m.group(1).replace("_", " ").replace("-", " ") + " ",
        demojized,
    )
    return WS_RE.sub(" ", result).strip()

def tokenize_social_text_demojized(text: str):
    text = clean_social_text(text)
    text = demojize_text(text)
    return tokenize_segment(text)

PRETOK_SEP = " "  # space works since tokens don't contain spaces

def pretokenize_column(texts, tokenizer_fn, desc="Tokenizing"):
    """Tokenize all texts once, return space-joined strings."""
    results = []
    for text in tqdm(texts, desc=desc):
        tokens = tokenizer_fn(str(text))
        results.append(PRETOK_SEP.join(tokens))
    return results

demo_text = "Great another Monday 🙄🔥 love it 🐻❤️"
print("Demo text:", demo_text)
print()
print("Tokenizer 1 (raw emoji):")
print(" ", tokenize_social_text(demo_text))
print()
print("Tokenizer 2 (demojized):")
print(" ", tokenize_social_text_demojized(demo_text))

def word_count_no_emoji(text: str) -> int:
    text = strip_emojis(clean_social_text(text))
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))

def sentiment_sign(x: float, eps: float = 0.05) -> int:
    if x > eps:
        return 1
    if x < -eps:
        return -1
    return 0

def max_consecutive_same(items):
    if not items:
        return 0
    best = 1
    cur = 1
    for i in range(1, len(items)):
        if items[i] == items[i - 1]:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return best

def extract_method2_features(text: str):
    text = clean_social_text(text)
    matches = extract_emoji_matches(text)

    emojis_raw = [m["emoji"] for m in matches]
    emojis_norm = [normalize_emoji_char(e) for e in emojis_raw]
    starts = [m["match_start"] for m in matches]

    text_len = max(len(text), 1)
    rel_positions = [s / text_len for s in starts] if starts else []

    known_scores = [emoji2score[e] for e in emojis_norm if e in emoji2score]
    known_count = len(known_scores)
    emoji_count = len(emojis_norm)
    unique_count = len(set(emojis_norm))
    unknown_count = emoji_count - known_count

    if known_scores:
        emoji_sent_mean = float(np.mean(known_scores))
        emoji_sent_min = float(np.min(known_scores))
        emoji_sent_max = float(np.max(known_scores))
        emoji_sent_std = float(np.std(known_scores))
        emoji_sent_sum = float(np.sum(known_scores))
    else:
        emoji_sent_mean = 0.0
        emoji_sent_min = 0.0
        emoji_sent_max = 0.0
        emoji_sent_std = 0.0
        emoji_sent_sum = 0.0

    wc = word_count_no_emoji(text)
    total_tokens_approx = wc + emoji_count
    emoji_token_ratio = emoji_count / total_tokens_approx if total_tokens_approx > 0 else 0.0
    emoji_per_word = emoji_count / wc if wc > 0 else float(emoji_count)
    emoji_diversity_ratio = unique_count / emoji_count if emoji_count > 0 else 0.0
    max_consecutive_repetition = max_consecutive_same(emojis_norm)

    has_emoji = int(emoji_count > 0)
    emoji_at_start = int(any(p <= 0.33 for p in rel_positions))
    emoji_in_middle = int(any(0.33 < p < 0.66 for p in rel_positions))
    emoji_at_end = int(any(p >= 0.66 for p in rel_positions))

    emoji_first_pos = float(rel_positions[0]) if rel_positions else 0.0
    emoji_last_pos = float(rel_positions[-1]) if rel_positions else 0.0
    emoji_mean_pos = float(np.mean(rel_positions)) if rel_positions else 0.0
    emoji_position_std = float(np.std(rel_positions)) if rel_positions else 0.0

    text_only = strip_emojis(text)
    text_only = WS_RE.sub(" ", text_only).strip()
    text_sentiment_vader = vader.polarity_scores(text_only)["compound"] if text_only else 0.0

    sentiment_difference = text_sentiment_vader - emoji_sent_mean
    abs_sentiment_difference = abs(sentiment_difference)

    t_sign = sentiment_sign(text_sentiment_vader)
    e_sign = sentiment_sign(emoji_sent_mean)

    agreement_flag = int(t_sign == e_sign and t_sign != 0 and e_sign != 0)
    contradiction_flag = int(t_sign * e_sign == -1)
    amplification_flag = int(
        t_sign == e_sign and t_sign != 0 and
        abs(text_sentiment_vader) > 0.3 and
        abs(emoji_sent_mean) > 0.3
    )

    return {
        "has_emoji": has_emoji,
        "emoji_known_count": known_count,
        "emoji_unknown_count": unknown_count,
        "emoji_lexicon_coverage": known_count / emoji_count if emoji_count > 0 else 0.0,
        "emoji_sent_mean": emoji_sent_mean,
        "emoji_sent_min": emoji_sent_min,
        "emoji_sent_max": emoji_sent_max,
        "emoji_sent_std": emoji_sent_std,
        "emoji_sent_sum": emoji_sent_sum,
        "emoji_count": emoji_count,
        "unique_emoji_count": unique_count,
        "emoji_token_ratio": emoji_token_ratio,
        "emoji_per_word": emoji_per_word,
        "emoji_diversity_ratio": emoji_diversity_ratio,
        "max_consecutive_repetition": max_consecutive_repetition,
        "emoji_at_start": emoji_at_start,
        "emoji_in_middle": emoji_in_middle,
        "emoji_at_end": emoji_at_end,
        "emoji_first_pos": emoji_first_pos,
        "emoji_last_pos": emoji_last_pos,
        "emoji_mean_pos": emoji_mean_pos,
        "emoji_position_std": emoji_position_std,
        "text_sentiment_vader": text_sentiment_vader,
        "sentiment_difference": sentiment_difference,
        "abs_sentiment_difference": abs_sentiment_difference,
        "agreement_flag": agreement_flag,
        "contradiction_flag": contradiction_flag,
        "amplification_flag": amplification_flag,
    }

NUMERIC_FEATURES = [
    "has_emoji",
    "emoji_known_count",
    "emoji_unknown_count",
    "emoji_lexicon_coverage",
    "emoji_sent_mean",
    "emoji_sent_min",
    "emoji_sent_max",
    "emoji_sent_std",
    "emoji_sent_sum",
    "emoji_count",
    "unique_emoji_count",
    "emoji_token_ratio",
    "emoji_per_word",
    "emoji_diversity_ratio",
    "max_consecutive_repetition",
    "emoji_at_start",
    "emoji_in_middle",
    "emoji_at_end",
    "emoji_first_pos",
    "emoji_last_pos",
    "emoji_mean_pos",
    "emoji_position_std",
    "text_sentiment_vader",
    "sentiment_difference",
    "abs_sentiment_difference",
    "agreement_flag",
    "contradiction_flag",
    "amplification_flag",
]

print("\nAll preprocessing functions defined.")

In [ ]:
# Batch VADER calls by pre-computing text-only versions, then apply VADER once
print("Extracting Method 2 features...")
feature_df = work_df["Text"].progress_apply(extract_method2_features).apply(pd.Series)

print("\nPre-tokenizing with raw-emoji tokenizer...")
pretok_raw = pretokenize_column(
    work_df["Text"].values, tokenize_social_text, desc="Tokenizer 1 (raw)"
)

print("Pre-tokenizing with demojized tokenizer...")
pretok_demoji = pretokenize_column(
    work_df["Text"].values, tokenize_social_text_demojized, desc="Tokenizer 2 (demoji)"
)

model_df = pd.concat(
    [work_df[["Text", "label"]].reset_index(drop=True), feature_df.reset_index(drop=True)],
    axis=1,
)
model_df["Text_pretok"] = pretok_raw
model_df["Text_pretok_demoji"] = pretok_demoji

print(f"\nModel dataframe shape: {model_df.shape}")
print(f"\nSample pre-tokenized text:")
print(f"  Raw text:   {model_df['Text'].iloc[0][:80]}...")
print(f"  Pretok:     {model_df['Text_pretok'].iloc[0][:80]}...")
print(f"  Demojized:  {model_df['Text_pretok_demoji'].iloc[0][:80]}...")
display(model_df.head())

### 1.7) Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X = model_df[["Text", "Text_pretok", "Text_pretok_demoji"] + NUMERIC_FEATURES].copy()
y = model_df["label"].astype(int).to_numpy()

X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train pool shape:", X_train_pool.shape)
print("Test shape      :", X_test.shape)

print("\nTrain-pool label distribution:")
print(pd.Series(y_train_pool).value_counts().sort_index())
print(pd.Series(y_train_pool).value_counts(normalize=True).sort_index())

print("\nHeld-out test label distribution:")
print(pd.Series(y_test).value_counts().sort_index())
print(pd.Series(y_test).value_counts(normalize=True).sort_index())

## Part 2–4 — Define all ML pipelines (Methods 1, 2, 2b)

**8 models** total:

| Key | Method | Text Column | Emoji Features | Classifier |
|-----|--------|-------------|----------------|------------|
| `baseline_lr` | 1 | `Text_pretok` (raw emoji) | No | LR |
| `baseline_rf` | 1 | `Text_pretok` | No | RF |
| `baseline_xgb` | 1 | `Text_pretok` | No | XGB |
| `method2_lr` | 2 | `Text_pretok` | Yes (28) | LR |
| `method2_rf` | 2 | `Text_pretok` | Yes (28) | RF |
| `method2_xgb` | 2 | `Text_pretok` | Yes (28) | XGB |
| `method2b_lr` | 2b | `Text_pretok_demoji` | Yes (28) | LR |
| `method2b_xgb` | 2b | `Text_pretok_demoji` | Yes (28) | XGB |

**All TF-IDF vectorizers use `str.split`** since text is already pre-tokenized.
No custom tokenizer runs during training — just string splitting.

In [ ]:
import os
import subprocess
import warnings
from collections import OrderedDict

import numpy as np
from joblib import Memory
from scipy.stats import randint, uniform, loguniform
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    make_scorer,
    precision_recall_fscore_support,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)

USE_GPU_IF_AVAILABLE = True
FAST_TREE_MODE = True

LR_SEARCH_N_JOBS = 2
TREE_SEARCH_N_JOBS = 1
RF_N_JOBS = -1
XGB_CPU_N_JOBS = 2

TREE_TFIDF_MAX_FEATURES = 20000 if FAST_TREE_MODE else 30000
TREE_TFIDF_MIN_DF = 5 if FAST_TREE_MODE else 3
TREE_SVD_COMPONENTS = 120 if FAST_TREE_MODE else 180
TREE_RF_N_ITER = 4 if FAST_TREE_MODE else 8
TREE_XGB_N_ITER = 6 if FAST_TREE_MODE else 10

CACHE_DIR = "/content/pipeline_cache_fastgpu"
os.makedirs(CACHE_DIR, exist_ok=True)
PIPELINE_MEMORY = Memory(CACHE_DIR, verbose=0)

def detect_gpu():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"], text=True).strip()
        return True, out
    except Exception as exc:
        return False, str(exc)

GPU_AVAILABLE, GPU_INFO = detect_gpu()
XGB_DEVICE = "cuda" if (USE_GPU_IF_AVAILABLE and GPU_AVAILABLE) else "cpu"

neg_count = int((y_train_pool == 0).sum())
pos_count = int((y_train_pool == 1).sum())
BASE_SCALE_POS_WEIGHT = float(neg_count / max(pos_count, 1))

print("GPU available:", GPU_AVAILABLE)
print("XGB device   :", XGB_DEVICE)
print("GPU info     :", GPU_INFO.splitlines()[0] if GPU_INFO else "N/A")
print(f"\nSpeed optimizations:")
print(f"  - Pre-tokenized text → TF-IDF uses str.split (no custom tokenizer during CV)")
print(f"  - LR uses class_weight='balanced' instead of RandomOverSampler (no data inflation)")
print(f"  - LR max_iter reduced to 2000 (was 8000 — rarely needs >1000)")
print(f"  - LR CV folds run in parallel (n_jobs={LR_SEARCH_N_JOBS})")
print(f"  - Tree models use Pipeline caching + SVD dimensionality reduction")

def make_tfidf_vectorizer(**overrides):
    params = dict(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        max_features=50000,
        sublinear_tf=True,
        dtype=np.float32,
    )
    params.update(overrides)
    return TfidfVectorizer(**params)

def make_numeric_transformer():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler", StandardScaler()),
    ])

def make_linear_preprocessor(use_numeric: bool, text_col: str = "Text_pretok"):
    transformers = [
        ("text", make_tfidf_vectorizer(), text_col),
    ]
    if use_numeric:
        transformers.append(("num", make_numeric_transformer(), NUMERIC_FEATURES))
    return ColumnTransformer(transformers=transformers, remainder="drop")

def make_tree_text_branch():
    return Pipeline(
        steps=[
            ("tfidf", make_tfidf_vectorizer(
                ngram_range=(1, 2),
                min_df=TREE_TFIDF_MIN_DF,
                max_df=0.98,
                max_features=TREE_TFIDF_MAX_FEATURES,
                sublinear_tf=True,
            )),
            ("svd", TruncatedSVD(
                n_components=TREE_SVD_COMPONENTS,
                algorithm="randomized",
                n_iter=5,
                random_state=RANDOM_STATE,
            )),
            ("scaler", StandardScaler()),
        ],
        memory=PIPELINE_MEMORY,
    )

def make_tree_preprocessor(use_numeric: bool, text_col: str = "Text_pretok"):
    transformers = [
        ("text", make_tree_text_branch(), text_col),
    ]
    if use_numeric:
        transformers.append(("num", make_numeric_transformer(), NUMERIC_FEATURES))
    return ColumnTransformer(transformers=transformers, remainder="drop")

def to_float32(X):
    return X.astype(np.float32)

def make_logistic_pipeline(use_numeric: bool, text_col: str = "Text_pretok"):
    return Pipeline(
        steps=[
            ("preprocess", make_linear_preprocessor(use_numeric=use_numeric, text_col=text_col)),
            ("clf", LogisticRegression(
                solver="saga",
                max_iter=2000,   # was 8000 — SAGA converges well within 1000 on this data
                tol=1e-3,        # slightly relaxed tolerance for speed
                random_state=RANDOM_STATE,
            )),
        ],
        memory=PIPELINE_MEMORY,
    )

def make_rf_pipeline(use_numeric: bool, text_col: str = "Text_pretok"):
    return Pipeline(
        steps=[
            ("preprocess", make_tree_preprocessor(use_numeric=use_numeric, text_col=text_col)),
            ("to_float32", FunctionTransformer(to_float32, accept_sparse=True)),
            ("clf", RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=RF_N_JOBS,
                class_weight="balanced_subsample",
            )),
        ],
        memory=PIPELINE_MEMORY,
    )

def make_xgb_pipeline(use_numeric: bool, text_col: str = "Text_pretok"):
    xgb_kwargs = dict(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device=XGB_DEVICE,
        random_state=RANDOM_STATE,
        verbosity=0,
    )
    if XGB_DEVICE == "cuda":
        xgb_kwargs["sampling_method"] = "gradient_based"
        xgb_kwargs["max_bin"] = 256
        xgb_kwargs["n_jobs"] = 1
    else:
        xgb_kwargs["n_jobs"] = XGB_CPU_N_JOBS

    return Pipeline(
        steps=[
            ("preprocess", make_tree_preprocessor(use_numeric=use_numeric, text_col=text_col)),
            ("to_float32", FunctionTransformer(to_float32, accept_sparse=True)),
            ("clf", XGBClassifier(**xgb_kwargs)),
        ],
        memory=PIPELINE_MEMORY,
    )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    "accuracy": "accuracy",
    "macro_f1": make_scorer(f1_score, average="macro", zero_division=0),
}

lr_params_l1l2 = {
    "preprocess__text__ngram_range": [(1, 1), (1, 2)],
    "preprocess__text__min_df": [2, 3, 5],
    "preprocess__text__max_df": [0.95, 0.98, 1.0],
    "preprocess__text__max_features": [20000, 40000, 60000],
    "preprocess__text__sublinear_tf": [True, False],
    "clf__penalty": ["l1", "l2"],
    "clf__C": loguniform(1e-3, 30.0),
    "clf__class_weight": [None, "balanced"],
}

lr_params_enet = {
    **{k: v for k, v in lr_params_l1l2.items() if "penalty" not in k},
    "clf__penalty": ["elasticnet"],
    "clf__C": loguniform(1e-3, 30.0),
    "clf__l1_ratio": uniform(0.05, 0.90),
    "clf__class_weight": [None, "balanced"],
}

rf_params = {
    "clf__n_estimators": randint(120, 281),
    "clf__max_depth": [None, 15, 30],
    "clf__min_samples_split": randint(2, 10),
    "clf__min_samples_leaf": randint(1, 4),
    "clf__max_features": ["sqrt", 0.5],
    "clf__bootstrap": [True],
    "clf__max_samples": [0.70, 0.85, None],
    "clf__class_weight": ["balanced", "balanced_subsample"],
}

xgb_params = {
    "clf__n_estimators": randint(120, 301),
    "clf__max_depth": randint(3, 7),
    "clf__learning_rate": loguniform(0.03, 0.20),
    "clf__subsample": uniform(0.65, 0.30),
    "clf__colsample_bytree": uniform(0.60, 0.30),
    "clf__min_child_weight": randint(1, 6),
    "clf__gamma": uniform(0.0, 0.40),
    "clf__reg_lambda": loguniform(1e-2, 5.0),
    "clf__scale_pos_weight": [1.0, BASE_SCALE_POS_WEIGHT, BASE_SCALE_POS_WEIGHT * 1.10],
}

MODEL_SPECS = OrderedDict([
    ("baseline_lr", {
        "label": "Method 1 — Baseline LR (TF-IDF, raw emoji tokens)",
        "group": "method1",
        "pipeline": make_logistic_pipeline(use_numeric=False, text_col="Text_pretok"),
        "param_distributions": [lr_params_l1l2, lr_params_enet],
        "n_iter": 14,
        "n_jobs": LR_SEARCH_N_JOBS,
    }),
    ("baseline_rf", {
        "label": "Method 1 — Baseline RF (TF-IDF → SVD)",
        "group": "method1",
        "pipeline": make_rf_pipeline(use_numeric=False, text_col="Text_pretok"),
        "param_distributions": rf_params,
        "n_iter": TREE_RF_N_ITER,
        "n_jobs": TREE_SEARCH_N_JOBS,
    }),
    ("baseline_xgb", {
        "label": f"Method 1 — Baseline XGB (TF-IDF → SVD, device={XGB_DEVICE})",
        "group": "method1",
        "pipeline": make_xgb_pipeline(use_numeric=False, text_col="Text_pretok"),
        "param_distributions": xgb_params,
        "n_iter": TREE_XGB_N_ITER,
        "n_jobs": TREE_SEARCH_N_JOBS,
    }),

    ("method2_lr", {
        "label": "Method 2 — LR (TF-IDF + 28 emoji features)",
        "group": "method2",
        "pipeline": make_logistic_pipeline(use_numeric=True, text_col="Text_pretok"),
        "param_distributions": [lr_params_l1l2, lr_params_enet],
        "n_iter": 16,
        "n_jobs": LR_SEARCH_N_JOBS,
    }),
    ("method2_rf", {
        "label": "Method 2 — RF (TF-IDF → SVD + emoji features)",
        "group": "method2",
        "pipeline": make_rf_pipeline(use_numeric=True, text_col="Text_pretok"),
        "param_distributions": rf_params,
        "n_iter": TREE_RF_N_ITER,
        "n_jobs": TREE_SEARCH_N_JOBS,
    }),
    ("method2_xgb", {
        "label": f"Method 2 — XGB (TF-IDF → SVD + emoji features, device={XGB_DEVICE})",
        "group": "method2",
        "pipeline": make_xgb_pipeline(use_numeric=True, text_col="Text_pretok"),
        "param_distributions": xgb_params,
        "n_iter": TREE_XGB_N_ITER,
        "n_jobs": TREE_SEARCH_N_JOBS,
    }),

    ("method2b_lr", {
        "label": "Method 2b — LR (demojized TF-IDF + 28 emoji features)",
        "group": "method2b",
        "pipeline": make_logistic_pipeline(use_numeric=True, text_col="Text_pretok_demoji"),
        "param_distributions": [lr_params_l1l2, lr_params_enet],
        "n_iter": 16,
        "n_jobs": LR_SEARCH_N_JOBS,
    }),
    ("method2b_xgb", {
        "label": f"Method 2b — XGB (demojized TF-IDF → SVD + emoji features, device={XGB_DEVICE})",
        "group": "method2b",
        "pipeline": make_xgb_pipeline(use_numeric=True, text_col="Text_pretok_demoji"),
        "param_distributions": xgb_params,
        "n_iter": TREE_XGB_N_ITER,
        "n_jobs": TREE_SEARCH_N_JOBS,
    }),
])

print(f"Defined {len(MODEL_SPECS)} model specs:")
for k, v in MODEL_SPECS.items():
    print(f"  {k:20s} → {v['label']}")

print(f"\nAll pipelines use str.split tokenizer on pre-tokenized columns.")
print(f"LR uses class_weight instead of RandomOverSampler — no data inflation.")

In [ ]:
def run_random_search(approach_key, spec, X_train, y_train):
    search = RandomizedSearchCV(
        estimator=spec["pipeline"],
        param_distributions=spec["param_distributions"],
        n_iter=spec["n_iter"],
        scoring=SCORING,
        refit="macro_f1",
        n_jobs=spec.get("n_jobs", 1),
        cv=cv,
        verbose=2,
        random_state=RANDOM_STATE,
        return_train_score=True,
    )

    print(f"\n{'=' * 20} {spec['label']} {'=' * 20}")
    search.fit(X_train, y_train)

    results = pd.DataFrame(search.cv_results_).sort_values(
        ["rank_test_macro_f1", "mean_test_macro_f1"],
        ascending=[True, False],
    )

    metric_cols = [
        "rank_test_macro_f1",
        "mean_train_accuracy",
        "mean_test_accuracy",
        "mean_train_macro_f1",
        "mean_test_macro_f1",
        "std_test_macro_f1",
        "mean_fit_time",
    ]
    param_cols = [c for c in results.columns if c.startswith("param_")]
    display_cols = metric_cols + param_cols[:12]

    print("\nBest CV macro-F1:", round(float(search.best_score_), 4))
    print("Best params:", search.best_params_)
    display(results[display_cols].head(5))

    return search, results

def summarize_cv_from_search(approach_key, spec, search):
    cv_results = search.cv_results_
    idx = search.best_index_
    return {
        "approach_key": approach_key,
        "approach": spec["label"],
        "group": spec["group"],
        "avg_train_accuracy": float(cv_results["mean_train_accuracy"][idx]),
        "avg_cv_test_accuracy": float(cv_results["mean_test_accuracy"][idx]),
        "avg_train_macro_f1": float(cv_results["mean_train_macro_f1"][idx]),
        "avg_cv_test_macro_f1": float(cv_results["mean_test_macro_f1"][idx]),
        "cv_macro_f1_std": float(cv_results["std_test_macro_f1"][idx]),
        "avg_fit_time_seconds": float(cv_results["mean_fit_time"][idx]),
        "best_params": search.best_params_,
    }

search_objects = {}
cv_results_tables = {}
cv_summary_rows = []

for approach_key, spec in MODEL_SPECS.items():
    search, results = run_random_search(approach_key, spec, X_train_pool, y_train_pool)
    search_objects[approach_key] = search
    cv_results_tables[approach_key] = results
    cv_summary_rows.append(summarize_cv_from_search(approach_key, spec, search))

In [ ]:
cv_summary_df = pd.DataFrame(cv_summary_rows).sort_values(
    ["group", "avg_cv_test_macro_f1"],
    ascending=[True, False],
)

display_cols = [
    "approach_key", "group",
    "avg_train_accuracy", "avg_cv_test_accuracy",
    "avg_train_macro_f1", "avg_cv_test_macro_f1",
    "cv_macro_f1_std", "avg_fit_time_seconds",
]

print("CV Summary (sorted by group then macro F1):")
display(cv_summary_df[display_cols].round(4))

In [ ]:
import matplotlib.pyplot as plt

def evaluate_classifier(model, X_eval, y_true, name="model"):
    y_pred = model.predict(X_eval)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0,
    )
    print(f"\n{'─'*50}")
    print(f"{name}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Macro F1:  {f1:.4f}")
    print(classification_report(
        y_true, y_pred,
        target_names=["Non-sarcastic", "Sarcastic"],
        digits=4,
    ))
    return {
        "approach_key": name,
        "accuracy": acc,
        "macro_f1": f1,
        "macro_precision": prec,
        "macro_recall": rec,
    }

holdout_rows = []
for approach_key, search in search_objects.items():
    row = evaluate_classifier(
        search.best_estimator_, X_test, y_test, name=approach_key,
    )
    holdout_rows.append(row)

holdout_summary_df = pd.DataFrame(holdout_rows).sort_values("macro_f1", ascending=False)

print("\n" + "=" * 60)
print("HELD-OUT TEST SUMMARY (sorted by macro F1)")
print("=" * 60)
display(holdout_summary_df.round(4))

In [ ]:
train_q25 = X_train_pool["emoji_count"].quantile(0.25)
train_q75 = X_train_pool["emoji_count"].quantile(0.75)

subset_masks = {
    "low_emoji": X_test["emoji_count"].to_numpy() <= train_q25,
    "high_emoji": X_test["emoji_count"].to_numpy() >= train_q75,
    "contradiction": X_test["contradiction_flag"].to_numpy() == 1,
}

subset_rows = []

for subset_name, mask in subset_masks.items():
    n = int(mask.sum())
    if n == 0:
        continue

    X_sub = X_test.loc[mask]
    y_sub = y_test[mask]

    for approach_key, search in search_objects.items():
        best_model = search.best_estimator_
        y_pred_sub = best_model.predict(X_sub)
        acc_sub = accuracy_score(y_sub, y_pred_sub)
        _, _, f1_sub, _ = precision_recall_fscore_support(
            y_sub, y_pred_sub, average="macro", zero_division=0,
        )
        subset_rows.append({
            "subset": subset_name,
            "n": n,
            "model": approach_key,
            "accuracy": round(acc_sub, 4),
            "macro_f1": round(f1_sub, 4),
        })

subset_eval_df = pd.DataFrame(subset_rows)

for sname in subset_masks:
    sub = subset_eval_df[subset_eval_df["subset"] == sname].sort_values("macro_f1", ascending=False)
    if len(sub) > 0:
        print(f"\nSubset: {sname} (n={sub['n'].iloc[0]})")
        display(sub[["model", "accuracy", "macro_f1"]])

In [ ]:
best_m2b_lr = search_objects["method2b_lr"].best_estimator_
preprocess = best_m2b_lr.named_steps["preprocess"]
clf = best_m2b_lr.named_steps["clf"]

feature_names = preprocess.get_feature_names_out()
coefs = clf.coef_.ravel()

coef_df = pd.DataFrame({
    "feature_name": feature_names,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
})

nonzero_coef_df = coef_df[coef_df["coefficient"] != 0].copy()
print("Non-zero coefficient count:", len(nonzero_coef_df), "out of", len(coef_df))

text_features = nonzero_coef_df[nonzero_coef_df["feature_name"].str.startswith("text__")]
top_positive = text_features.nlargest(20, "coefficient")[["feature_name", "coefficient"]]
top_negative = text_features.nsmallest(20, "coefficient")[["feature_name", "coefficient"]]

print("\nTop 20 features → SARCASTIC:")
display(top_positive)
print("\nTop 20 features → NON-SARCASTIC:")
display(top_negative)

emoji_feats = nonzero_coef_df[nonzero_coef_df["feature_name"].str.startswith("num__")]
print("\nEmoji feature coefficients:")
display(emoji_feats.sort_values("abs_coefficient", ascending=False))

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "vinai/bertweet-base"
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    print("Using fast tokenizer")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    print("Falling back to slow tokenizer")

sample = "Great another Monday 🙄🔥"
tokens = tokenizer.tokenize(sample)
print(f"Sample: {sample}")
print(f"Tokens: {tokens}")
print(f"Token count: {len(tokens)}")

In [ ]:
from datasets import Dataset

MAX_LENGTH = 128

# Re-split using original text for BERTweet (it needs raw text, not pre-tokenized)
all_text = model_df["Text"].values
all_label = model_df["label"].values

from sklearn.model_selection import train_test_split as tts

idx_trainval, idx_test = tts(
    np.arange(len(all_label)), test_size=0.10,
    stratify=all_label, random_state=RANDOM_STATE,
)
idx_train, idx_val = tts(
    idx_trainval, test_size=0.111,
    stratify=all_label[idx_trainval], random_state=RANDOM_STATE,
)

bert_train_df = pd.DataFrame({"text": all_text[idx_train], "label": all_label[idx_train]})
bert_val_df = pd.DataFrame({"text": all_text[idx_val], "label": all_label[idx_val]})
bert_test_df = pd.DataFrame({"text": all_text[idx_test], "label": all_label[idx_test]})

print(f"BERTweet splits — Train: {len(bert_train_df)}  Val: {len(bert_val_df)}  Test: {len(bert_test_df)}")

def make_hf_dataset(dataframe):
    ds = Dataset.from_pandas(dataframe[["text", "label"]].reset_index(drop=True))
    ds = ds.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        ),
        batched=True,
        batch_size=512,
    )
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

train_ds = make_hf_dataset(bert_train_df)
val_ds = make_hf_dataset(bert_val_df)
test_ds = make_hf_dataset(bert_test_df)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn

class_weights = compute_class_weight(
    "balanced", classes=np.array([0, 1]), y=bert_train_df["label"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights: {class_weights}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0,
    )
    return {"accuracy": acc, "macro_f1": f1, "macro_precision": prec, "macro_recall": rec}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True,
)
model.to(device)

training_args = TrainingArguments(
    output_dir="./bertweet_sarcoji",
    num_train_epochs=5,
    per_device_train_batch_size=48,  # T4 has 15GB — 48 fits comfortably
    per_device_eval_batch_size=96,   # eval doesn't store gradients → can go larger
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
    seed=RANDOM_STATE,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
train_result = trainer.train()
print(f"\nTraining complete. Best model loaded from checkpoint.")

In [ ]:
log_history = trainer.state.log_history

train_losses = [(e["step"], e["loss"]) for e in log_history if "loss" in e and "eval_loss" not in e]
eval_entries = [e for e in log_history if "eval_loss" in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, label="Train loss", alpha=0.7)
if eval_entries:
    eval_steps = [e["step"] for e in eval_entries]
    eval_losses = [e["eval_loss"] for e in eval_entries]
    axes[0].plot(eval_steps, eval_losses, "o-", label="Val loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()

if eval_entries:
    epochs = list(range(1, len(eval_entries) + 1))
    f1s = [e["eval_macro_f1"] for e in eval_entries]
    accs = [e["eval_accuracy"] for e in eval_entries]
    axes[1].plot(epochs, f1s, "s-", label="Macro F1")
    axes[1].plot(epochs, accs, "^-", label="Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_title("Validation Metrics per Epoch")
    axes[1].set_xticks(epochs)
    axes[1].legend()
    axes[1].set_ylim(0.5, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
test_output = trainer.predict(test_ds)
bert_test_preds = np.argmax(test_output.predictions, axis=-1)
bert_test_probs = torch.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()
bert_y_test = bert_test_df["label"].values

print("=" * 60)
print("HELD-OUT TEST RESULTS — BERTweet Fine-Tuned")
print("=" * 60)
print(classification_report(
    bert_y_test, bert_test_preds,
    target_names=["Non-sarcastic", "Sarcastic"],
    digits=4,
))

bert_test_acc = accuracy_score(bert_y_test, bert_test_preds)
_, _, bert_test_f1, _ = precision_recall_fscore_support(
    bert_y_test, bert_test_preds, average="macro", zero_division=0,
)
print(f"Accuracy:  {bert_test_acc:.4f}")
print(f"Macro F1:  {bert_test_f1:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(bert_y_test, bert_test_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Non-sarcastic", "Sarcastic"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("BERTweet — Held-Out Test Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import emoji as emoji_lib

def count_emojis(text):
    return sum(1 for ch in str(text) if emoji_lib.is_emoji(ch))

def get_text_sentiment_quick(text):
    stripped = "".join(ch for ch in str(text) if not emoji_lib.is_emoji(ch))
    stripped = URL_RE.sub("", stripped)
    stripped = MENTION_RE.sub("", stripped)
    stripped = HTML_RE.sub("", stripped)
    return vader.polarity_scores(stripped.strip())["compound"]

def get_emoji_sentiment_quick(text):
    scores = [emoji2score[ch] for ch in str(text) if ch in emoji2score]
    return np.mean(scores) if scores else 0.0

def compute_contradiction_flag_quick(text):
    ts = get_text_sentiment_quick(text)
    es = get_emoji_sentiment_quick(text)
    if abs(ts) < 0.05 or abs(es) < 0.01:
        return 0
    return int((ts > 0) != (es > 0))

bert_test_df = bert_test_df.copy()
bert_test_df["emoji_count"] = bert_test_df["text"].apply(count_emojis)
bert_test_df["contradiction_flag"] = bert_test_df["text"].apply(compute_contradiction_flag_quick)

q25 = bert_test_df["emoji_count"].quantile(0.25)
q75 = bert_test_df["emoji_count"].quantile(0.75)

bert_subset_masks = {
    "low_emoji":     bert_test_df["emoji_count"].values <= q25,
    "high_emoji":    bert_test_df["emoji_count"].values >= q75,
    "contradiction": bert_test_df["contradiction_flag"].values == 1,
}

bert_subset_rows = []
for name, mask in bert_subset_masks.items():
    n = int(mask.sum())
    if n == 0:
        continue
    y_sub = bert_y_test[mask]
    p_sub = bert_test_preds[mask]
    acc = accuracy_score(y_sub, p_sub)
    _, _, f1, _ = precision_recall_fscore_support(y_sub, p_sub, average="macro", zero_division=0)
    bert_subset_rows.append({"subset": name, "n": n, "accuracy": acc, "macro_f1": f1})
    print(f"  {name:20s}  n={n:5d}  acc={acc:.4f}  macro_f1={f1:.4f}")

bert_subset_rows.append({"subset": "overall", "n": len(bert_y_test),
                         "accuracy": bert_test_acc, "macro_f1": bert_test_f1})
bert_subset_df = pd.DataFrame(bert_subset_rows)
display(bert_subset_df)

In [ ]:
all_results = holdout_summary_df.copy()
all_results["method"] = all_results["approach_key"].map({
    "baseline_lr": "Method 1", "baseline_rf": "Method 1", "baseline_xgb": "Method 1",
    "method2_lr": "Method 2", "method2_rf": "Method 2", "method2_xgb": "Method 2",
    "method2b_lr": "Method 2b", "method2b_xgb": "Method 2b",
})

bert_row = pd.DataFrame([{
    "approach_key": "bertweet",
    "accuracy": round(bert_test_acc, 4),
    "macro_f1": round(bert_test_f1, 4),
    "macro_precision": np.nan,
    "macro_recall": np.nan,
    "method": "Method 3",
}])
overall_comparison_df = pd.concat([all_results, bert_row], ignore_index=True)
overall_comparison_df = overall_comparison_df.sort_values("macro_f1", ascending=False)

print("=" * 60)
print("CROSS-METHOD COMPARISON — All Models")
print("=" * 60)
display(overall_comparison_df[["approach_key", "method", "accuracy", "macro_f1"]].round(4))

fig, ax = plt.subplots(figsize=(12, 6))
colors = {
    "Method 1": "#73726c", "Method 2": "#3266ad",
    "Method 2b": "#e07b39", "Method 3": "#2a9d5c",
}
bars = ax.barh(
    overall_comparison_df["approach_key"],
    overall_comparison_df["macro_f1"],
    color=[colors.get(m, "#999") for m in overall_comparison_df["method"]],
)
ax.set_xlabel("Macro F1")
ax.set_title("All Methods — Held-Out Test Macro F1")
ax.set_xlim(0.75, 0.92)

for bar, val in zip(bars, overall_comparison_df["macro_f1"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=m) for m, c in colors.items()]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.show()

Cross-Dataset Evaluation: StockEmotions


In [ ]:
stock_path = kagglehub.dataset_download("goddiao/stockemotions")
print("Data located at:", stock_path)

stock_csvs = glob.glob(f"{stock_path}/**/*.csv", recursive=True)

stock_parts = []
for csv_path in sorted(stock_csvs):
    part = pd.read_csv(csv_path)
    part["_source_file"] = os.path.basename(csv_path)
    stock_parts.append(part)
    print(f"  {os.path.basename(csv_path)}: {len(part)} rows")

stock_raw = pd.concat(stock_parts, ignore_index=True)
print(f"\nCombined shape: {stock_raw.shape}")

In [ ]:
print("Available columns:", list(stock_raw.columns))

senti_col = None
for candidate in ["senti_label", "senti_lable", "sentiment", "Sentiment"]:
    if candidate in stock_raw.columns:
        senti_col = candidate
        break
if senti_col is None:
    raise ValueError(f"Cannot find sentiment column. Available: {list(stock_raw.columns)}")

text_col = None
for candidate in ["original", "text", "processed", "content", "tweet"]:
    if candidate in stock_raw.columns:
        text_col = candidate
        break
if text_col is None:
    raise ValueError(f"Cannot find text column. Available: {list(stock_raw.columns)}")

print(f"Using sentiment column: '{senti_col}'")
print(f"Using text column: '{text_col}'")
print(f"\nSentiment distribution (raw):")
print(stock_raw[senti_col].value_counts())

stock_df = stock_raw.copy()
stock_df["sentiment"] = stock_df[senti_col].astype(str).str.strip().str.lower()
stock_df = stock_df[stock_df["sentiment"].isin(["bullish", "bearish"])].copy()
stock_df["label"] = (stock_df["sentiment"] == "bearish").astype(int)
stock_df["text"] = stock_df[text_col].astype(str)

stock_df = stock_df[stock_df["text"].str.strip().str.len() > 0].copy()
stock_df = stock_df.drop_duplicates(subset="text").reset_index(drop=True)

def has_emoji_fn(text: str) -> bool:
    return len(emoji.emoji_list(str(text))) > 0

stock_df["has_emoji"] = stock_df["text"].apply(has_emoji_fn)

print(f"\nAfter binary filter + dedup: {len(stock_df)} posts")
print(f"Emoji-containing: {stock_df['has_emoji'].sum()} ({stock_df['has_emoji'].mean():.1%})")
print(f"\nLabel distribution:")
print(stock_df["label"].value_counts().rename({0: "bullish", 1: "bearish"}))

stock_emoji_df = stock_df[stock_df["has_emoji"]].reset_index(drop=True)
print(f"\nEmoji subset: {len(stock_emoji_df)} posts")

In [ ]:
def build_stock_model_df(df, text_col_name="text", label_col="label"):
    """Extract features + pre-tokenize for StockEmotions."""
    feat_df = df[text_col_name].progress_apply(extract_method2_features).apply(pd.Series)
    for col in NUMERIC_FEATURES:
        if col not in feat_df.columns:
            feat_df[col] = 0.0

    pretok_raw = pretokenize_column(
        df[text_col_name].values, tokenize_social_text, desc="Stock pretok (raw)"
    )
    pretok_demoji = pretokenize_column(
        df[text_col_name].values, tokenize_social_text_demojized, desc="Stock pretok (demoji)"
    )

    out = pd.concat([
        df[[text_col_name, label_col]].reset_index(drop=True).rename(columns={text_col_name: "Text"}),
        feat_df[NUMERIC_FEATURES].reset_index(drop=True),
    ], axis=1)
    out["Text_pretok"] = pretok_raw
    out["Text_pretok_demoji"] = pretok_demoji
    return out

print("Building features for FULL stock dataset...")
stock_model_full = build_stock_model_df(stock_df, text_col_name="text", label_col="label")

print(f"\nBuilding features for EMOJI-ONLY stock subset...")
stock_model_emoji = build_stock_model_df(stock_emoji_df, text_col_name="text", label_col="label")

print(f"\nFull model df shape:  {stock_model_full.shape}")
print(f"Emoji model df shape: {stock_model_emoji.shape}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV as RSC
from scipy.stats import loguniform as lu

# Use str.split on pre-tokenized columns — same speedup as main pipeline
def make_tfidf_stock():
    return TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        max_features=30000,
        sublinear_tf=True,
        dtype=np.float32,
    )

def make_stock_preprocessor(use_numeric: bool, text_col: str = "Text_pretok"):
    transformers = [("text", Pipeline([("tfidf", make_tfidf_stock())]), text_col)]
    if use_numeric:
        transformers.append(("num", make_numeric_transformer(), NUMERIC_FEATURES))
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=1.0)

def make_stock_lr_pipeline(use_numeric: bool, text_col: str = "Text_pretok"):
    return Pipeline([
        ("preprocess", make_stock_preprocessor(use_numeric, text_col)),
        ("clf", LogisticRegression(max_iter=2000, tol=1e-3, random_state=RANDOM_STATE)),
    ])

STOCK_LR_PARAMS = {
    "clf__solver": ["liblinear"],
    "clf__penalty": ["l1", "l2"],
    "clf__C": lu(1e-2, 10.0),
    "clf__class_weight": [None, "balanced"],
}

stock_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def train_stock_models(model_df_in, dataset_name, label_names=None):
    if label_names is None:
        label_names = ["bullish", "bearish"]

    X = model_df_in[["Text", "Text_pretok", "Text_pretok_demoji"] + NUMERIC_FEATURES].copy()
    y_arr = model_df_in["label"].astype(int).values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_arr, test_size=0.10, stratify=y_arr, random_state=RANDOM_STATE,
    )
    print(f"\n{'#'*70}")
    print(f"  {dataset_name}")
    print(f"  Train: {len(y_tr)}  |  Test: {len(y_te)}")
    print(f"{'#'*70}")

    results = {}
    for approach, use_num, tcol in [
        ("baseline_lr", False, "Text_pretok"),
        ("method2_lr",  True,  "Text_pretok"),
        ("method2b_lr", True,  "Text_pretok_demoji"),
    ]:
        print(f"\n--- {approach} ---")
        search = RSC(
            estimator=make_stock_lr_pipeline(use_numeric=use_num, text_col=tcol),
            param_distributions=STOCK_LR_PARAMS,
            n_iter=8,
            scoring=SCORING,
            refit="macro_f1",
            cv=stock_cv,
            verbose=1,
            random_state=RANDOM_STATE,
            return_train_score=True,
            n_jobs=2,  # LR is single-threaded → parallelize CV folds
        )
        search.fit(X_tr, y_tr)

        best_model = search.best_estimator_
        y_pred = best_model.predict(X_te)
        acc = accuracy_score(y_te, y_pred)
        _, _, f1, _ = precision_recall_fscore_support(y_te, y_pred, average="macro", zero_division=0)

        print(f"Best CV macro-F1: {search.best_score_:.4f}")
        print(f"Holdout — Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")
        print(classification_report(y_te, y_pred, target_names=label_names, digits=4))

        results[approach] = {
            "search": search,
            "accuracy": acc,
            "macro_f1": f1,
            "X_test": X_te,
            "y_test": y_te,
        }

    summary = pd.DataFrame([
        {"model": k, "accuracy": v["accuracy"], "macro_f1": v["macro_f1"]}
        for k, v in results.items()
    ])

    print(f"\n{'='*60}")
    print(f"{dataset_name} — COMPARISON")
    print(f"{'='*60}")
    display(summary)

    lift_m2 = results["method2_lr"]["macro_f1"] - results["baseline_lr"]["macro_f1"]
    lift_m2b = results["method2b_lr"]["macro_f1"] - results["baseline_lr"]["macro_f1"]
    print(f"\nMethod 2  lift: {lift_m2:+.4f} ({lift_m2*100:+.2f} pp)")
    print(f"Method 2b lift: {lift_m2b:+.4f} ({lift_m2b*100:+.2f} pp)")

    return results, summary

print("Training pipeline defined.")

In [ ]:
stock_results_full, stock_summary_full = train_stock_models(
    stock_model_full,
    dataset_name="StockEmotions (all posts)",
    label_names=["bullish", "bearish"],
)

In [ ]:
if len(stock_model_emoji) >= 100:
    stock_results_emoji, stock_summary_emoji = train_stock_models(
        stock_model_emoji,
        dataset_name="StockEmotions (emoji-only)",
        label_names=["bullish", "bearish"],
    )
else:
    print(f"Emoji subset too small ({len(stock_model_emoji)} rows) — skipping.")

Error analysis : which posts does Method 2b flip

In [ ]:
def error_analysis(results, dataset_name, label_names=None):
    if label_names is None:
        label_names = ["class_0", "class_1"]

    X_te = results["baseline_lr"]["X_test"]
    y_te = results["baseline_lr"]["y_test"]

    base_pred = results["baseline_lr"]["search"].best_estimator_.predict(X_te)
    m2b_pred = results["method2b_lr"]["search"].best_estimator_.predict(X_te)

    base_correct = (base_pred == y_te)
    m2b_correct = (m2b_pred == y_te)

    fixed_mask = (~base_correct) & m2b_correct
    broken_mask = base_correct & (~m2b_correct)

    n_fixed = int(fixed_mask.sum())
    n_broken = int(broken_mask.sum())

    print(f"\n{'='*60}")
    print(f"ERROR ANALYSIS: {dataset_name}")
    print(f"{'='*60}")
    print(f"Baseline errors:          {(~base_correct).sum()}")
    print(f"Method 2b errors:         {(~m2b_correct).sum()}")
    print(f"Fixed by demojize:        {n_fixed}")
    print(f"Broken by demojize:       {n_broken}")
    print(f"Net improvement:          {n_fixed - n_broken:+d} posts")

    emoji_cols = ["emoji_count", "contradiction_flag", "amplification_flag",
                  "abs_sentiment_difference", "emoji_sent_mean"]
    available_cols = [c for c in emoji_cols if c in X_te.columns]

    groups = {"all_test": np.ones(len(y_te), dtype=bool)}
    if n_fixed >= 5:
        groups["fixed_by_m2b"] = fixed_mask.values if hasattr(fixed_mask, 'values') else fixed_mask
    if n_broken >= 5:
        groups["broken_by_m2b"] = broken_mask.values if hasattr(broken_mask, 'values') else broken_mask

    profile_rows = []
    for gname, gmask in groups.items():
        row = {"group": gname, "n": int(gmask.sum())}
        for col in available_cols:
            row[f"mean_{col}"] = float(X_te.loc[gmask, col].mean())
        profile_rows.append(row)

    profile_df = pd.DataFrame(profile_rows)
    print(f"\nEmoji profile comparison:")
    display(profile_df)

    if n_fixed > 0:
        print(f"\nExample posts FIXED by Method 2b (baseline wrong → demojize right):")
        fixed_idx = X_te.index[fixed_mask]
        for i, idx in enumerate(fixed_idx[:8]):
            true_label = label_names[y_te[X_te.index.get_loc(idx)]]
            base_label = label_names[base_pred[X_te.index.get_loc(idx)]]
            text = X_te.loc[idx, "Text"]
            print(f"  [{i+1}] true={true_label}, base_pred={base_label}")
            print(f"      {text[:120]}")
            print(f"      demojized: {demojize_text(text)[:120]}")
            print()

error_analysis(stock_results_full, "StockEmotions (all posts)", ["bullish", "bearish"])

In [ ]:
grand_rows = []

for _, row in holdout_summary_df.iterrows():
    if row["approach_key"] in ["baseline_lr", "method2_lr", "method2b_lr"]:
        grand_rows.append({
            "dataset": "SarcOji (sarcasm)",
            "approach": row["approach_key"],
            "macro_f1": row["macro_f1"],
        })

for _, row in stock_summary_full.iterrows():
    grand_rows.append({
        "dataset": "StockEmotions (all)",
        "approach": row["model"],
        "macro_f1": row["macro_f1"],
    })

if 'stock_summary_emoji' in dir():
    for _, row in stock_summary_emoji.iterrows():
        grand_rows.append({
            "dataset": "StockEmotions (emoji-only)",
            "approach": row["model"],
            "macro_f1": row["macro_f1"],
        })

grand_df = pd.DataFrame(grand_rows)
pivot = grand_df.pivot(index="dataset", columns="approach", values="macro_f1")

if "baseline_lr" in pivot.columns:
    if "method2_lr" in pivot.columns:
        pivot["lift_m2_pp"] = (pivot["method2_lr"] - pivot["baseline_lr"]) * 100
    if "method2b_lr" in pivot.columns:
        pivot["lift_m2b_pp"] = (pivot["method2b_lr"] - pivot["baseline_lr"]) * 100

print("GRAND SUMMARY — Macro F1 (Baseline vs Method 2 vs Method 2b)")
print("=" * 70)
display(pivot.round(4))

if "lift_m2b_pp" in pivot.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    datasets = pivot.index.tolist()
    x = np.arange(len(datasets))
    w = 0.25
    ax.bar(x - w, pivot.get("baseline_lr", 0), w, label="Baseline", color="#73726c")
    ax.bar(x,     pivot.get("method2_lr", 0),  w, label="Method 2", color="#3266ad")
    ax.bar(x + w, pivot.get("method2b_lr", 0), w, label="Method 2b (demojized)", color="#e07b39")
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("Macro F1")
    ax.set_title("Baseline vs Method 2 vs Method 2b across datasets")
    ax.legend()
    ax.set_ylim(0.5, 1.0)
    plt.tight_layout()
    plt.show()

Save model

In [ ]:
import joblib
from pathlib import Path

ARTIFACT_DIR = Path("/content/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

bundle = {
    "models": {k: v.best_estimator_ for k, v in search_objects.items()},
    "cv_summary_df": cv_summary_df,
    "holdout_summary_df": holdout_summary_df,
    "overall_comparison_df": overall_comparison_df,
    "subset_eval_df": subset_eval_df,
    "numeric_features": NUMERIC_FEATURES,
    "emoji2score": emoji2score,
    "model_specs": {
        k: {"label": v["label"], "group": v["group"]}
        for k, v in MODEL_SPECS.items()
    },
}

bundle_path = ARTIFACT_DIR / "sarcoji_model_suite_balanced_cv.joblib"
joblib.dump(bundle, bundle_path, compress=3)
print(f"ML models saved to {bundle_path}")

model_save_path = ARTIFACT_DIR / "bertweet_sarcoji_finetuned"
trainer.save_model(str(model_save_path))
tokenizer.save_pretrained(str(model_save_path))
print(f"BERTweet saved to {model_save_path}")

print("\nAll artifacts saved.")

## For front end usage:

In [ ]:
def build_inference_frame(texts):
    """Build X matrix for prediction. Pre-tokenizes on the fly (single-text is fast)."""
    text_series = pd.Series(texts, name="Text").astype(str)
    feat_df = text_series.apply(extract_method2_features).apply(pd.Series)
    for col in NUMERIC_FEATURES:
        if col not in feat_df.columns:
            feat_df[col] = 0.0

    pretok = [PRETOK_SEP.join(tokenize_social_text(t)) for t in texts]
    pretok_demoji = [PRETOK_SEP.join(tokenize_social_text_demojized(t)) for t in texts]

    out = pd.concat([
        text_series.reset_index(drop=True),
        feat_df[NUMERIC_FEATURES].reset_index(drop=True),
    ], axis=1)
    out["Text_pretok"] = pretok
    out["Text_pretok_demoji"] = pretok_demoji
    return out

def predict_sarcasm(text: str, model_key: str = "method2b_lr"):
    X = build_inference_frame([text])
    model = search_objects[model_key].best_estimator_
    pred = model.predict(X)[0]
    prob = model.predict_proba(X)[0]
    features = extract_method2_features(text)
    return {
        "label": int(pred),
        "probability": {0: float(prob[0]), 1: float(prob[1])},
        "engineered_features": features,
        "demojized_text": demojize_text(text),
    }

test_text = "Oh great another Monday 🙄🔥"
result = predict_sarcasm(test_text)
print(f"Text: {test_text}")
print(f"Demojized: {result['demojized_text']}")
print(f"Prediction: {'Sarcastic' if result['label'] == 1 else 'Non-sarcastic'}")
print(f"Probabilities: {result['probability']}")

In [ ]:
import gradio as gr

MODEL_LABEL_MAP = {k: spec["label"] for k, spec in MODEL_SPECS.items()}

def gradio_predict(text, model_key):
    out = predict_sarcasm(text, model_key=model_key)
    feat_preview = pd.DataFrame([out["engineered_features"]])[[
        "emoji_count",
        "unique_emoji_count",
        "emoji_sent_mean",
        "text_sentiment_vader",
        "sentiment_difference",
        "contradiction_flag",
        "amplification_flag",
    ]]
    label_scores = {
        "Non-sarcastic": out["probability"][0],
        "Sarcastic": out["probability"][1],
    }
    demojized = out["demojized_text"]
    return label_scores, feat_preview, demojized

demo = gr.Interface(
    fn=gradio_predict,
    inputs=[
        gr.Textbox(label="Enter text (with emojis!)", lines=3,
                    placeholder="Oh great another Monday 🙄🔥"),
        gr.Dropdown(choices=list(MODEL_LABEL_MAP.keys()),
                    value="method2b_lr", label="Model"),
    ],
    outputs=[
        gr.Label(label="Prediction"),
        gr.Dataframe(label="Key emoji features"),
        gr.Textbox(label="Demojized text"),
    ],
    title="Emoji-Aware Sarcasm Detector",
    description="Compare Methods 1, 2, 2b across different models. Method 2b uses demojized emoji tokens.",
)

demo.launch(share=True)